# MyDigitalTwin — K-Means Clustering

**Objectif** : Construire des clusters de centres d'intérêts data-driven pour remplacer le keyword matching de la home page.

**Deux axes** :
- **Partie A — Content Clustering** : TF-IDF sur tous les textes (titres YouTube, recherches Google, artistes Spotify, films Netflix, produits Amazon...) → K-Means → top termes par cluster
- **Partie B — Behavioral Clustering** : heure, jour, plateforme → K-Means → profils comportementaux
- **Partie C — Fusion** : combiner les deux pour créer `interest_profiles` (lu par la home page)

**Outputs Delta** :
- `warehouse/content_clusters`
- `warehouse/behavioral_clusters`
- `warehouse/interest_profiles`

In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, ArrayType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-Clustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Paths Docker vs local
if os.path.exists("/opt/spark/warehouse"):
    WAREHOUSE = "/opt/spark/warehouse"
else:
    # Windows local (chemin absolu)
    WAREHOUSE = os.path.abspath(os.path.join(os.getcwd(), "../../../scripts/notebooks", "..", "..", "warehouse"))
    if not os.path.exists(WAREHOUSE):
        WAREHOUSE = "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

---
## PARTIE A — Content Clustering

On rassemble tout le texte consommé (titres, requêtes, artistes, produits) en une seule colonne `text`, puis on applique TF-IDF + KMeans.

In [ ]:
# ── A1. CHARGEMENT DES SOURCES TEXTE ──────────────────────────────────────────

def read_table(table_name):
    """Lit une table Delta (parquet files) depuis le warehouse."""
    path = os.path.join(WAREHOUSE, table_name)
    return spark.read.parquet(path)

# YouTube — titres de vidéos regardées
yt_watch = read_table("youtube_watch")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("youtube").alias("platform"),
            F.col("interaction_weight").alias("weight"))

# YouTube Searches — requêtes
yt_search = read_table("youtube_searches")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("youtube").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Google Searches — requêtes
g_search = read_table("google_searches")     .select(F.col("query").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("google").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Spotify — artiste + titre
spotify = read_table("spotify_streams") \
      .select(F.col("artistName").alias("text"),
              F.col("listen_hour").alias("hour"),
              F.col("listen_weekday").alias("weekday"),
              F.lit("spotify").alias("platform"),
              F.col("interaction_weight").alias("weight")) \
      .dropDuplicates(["text"])

# Netflix — titre de l'émission
netflix = read_table("netflix_views")     .select(F.col("show_title").alias("text"),
            F.lit(21).cast(IntegerType()).alias("hour"),
            F.col("watch_weekday").alias("weekday"), F.lit("netflix").alias("platform"),
            F.col("interaction_weight").alias("weight"))

# Amazon — nom produit + catégorie
amazon = read_table("amazon_orders") \
      .select(F.col("category").alias("text"),   # ← plus de product_name
              F.col("order_hour").alias("hour"),
              F.col("order_weekday").alias("weekday"),
              F.lit("amazon").alias("platform"),
              F.col("interaction_weight").alias("weight")) \
      .dropDuplicates(["text"])

# Apple App Installs — pas de interaction_weight dans cette table
apple = read_table("apple_app_installs")     .withColumn("text", F.concat_ws(" ", F.col("app_name"), F.col("category")))     .select(F.col("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("apple").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Google Chrome — titres de pages visitées
chrome = read_table("google_chrome")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("chrome").alias("platform"),
            F.lit(1.5).cast(FloatType()).alias("weight"))

# ── Union de toutes les sources
all_text = yt_watch.union(yt_search).union(g_search).union(spotify)     .union(netflix).union(amazon).union(apple).union(chrome)

# Nettoyage : supprimer les nulls et les textes trop courts
all_text = all_text.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 3)
)

print(f"Total events : {all_text.count():,}")
all_text.show(5, truncate=50)


In [ ]:
# ── A1b. NETTOYAGE ANTI-POLLUTION ─────────────────────────────────────────────
import re
from pyspark.sql.window import Window

# 1. Mots néerlandais courants (pages AliExpress BE, Chrome néerlandais)
NL_STOPWORDS = [
    "van", "voor", "met", "een", "het", "de", "en", "naar", "bij", "op",
    "uit", "als", "ook", "maar", "zijn", "wordt", "wordt", "meer", "alle",
    "winkelen", "populaire", "speelgoed"
]

# 2. Regex : patterns d'ads YouTube (codes internes INH/ING/Belfius)
AD_PATTERN = re.compile(
    r'(INH|VID\s+16x9|16x9\s+\d|TrustNew|BrandBranch|DailyBanking|'
    r'UnpackingMetal|ContactlessComp|FreeBankAccount)',
    re.IGNORECASE
)

def is_clean(text):
    if not text:
        return False
    t = text.lower()
    # Filtre 1 : ads YouTube (patterns internes)
    if AD_PATTERN.search(text):
        return False
    # Filtre 2 : texte majoritairement néerlandais (>2 mots NL sur les 6 premiers)
    words = t.split()[:8]
    if sum(1 for w in words if w in NL_STOPWORDS) >= 2:
        return False
    # Filtre 3 : texte trop long = description produit Amazon (> 120 chars)
    if len(text) > 120:
        return False
    if "gmail" in t or "mail.google" in t or "@" in t:
        return False
    return True

is_clean_udf = F.udf(is_clean, "boolean")

before = all_text.count()
all_text = all_text.filter(is_clean_udf(F.col("text")))

# Filtre 4 : dédupliquer les textes identiques (ads répétées à l'identique)
all_text = all_text.dropDuplicates(["text"])

w = Window.partitionBy("platform").orderBy(F.rand(seed=42))
all_text = all_text.withColumn("rn", F.row_number().over(w)) \
                     .filter(F.col("rn") <= 2000) \
                     .drop("rn")

after = all_text.count()
print(f"Avant nettoyage : {before:,} | Après : {after:,} | Supprimés : {before - after:,}")
all_text.groupBy("platform").count().orderBy(F.desc("count")).show()

In [ ]:
# ── A2. PIPELINE TF-IDF ──────────────────────────────────────────────────────
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, Normalizer

STOPWORDS_FR = [
    "le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "a", "au",
    "pour", "par", "sur", "avec", "dans", "qui", "que", "se", "il", "elle",
    "on", "je", "tu", "nous", "vous", "ils", "elles", "est", "sont",
    "ce", "son", "sa", "ses", "mon", "ma", "mes", "ton", "ta", "tes",
    "this", "the", "of", "in", "to", "and", "is", "for", "with",
    "official", "video", "youtube", "episode", "season", "saison"
]

tokenizer  = Tokenizer(inputCol="text", outputCol="words_raw")
remover    = StopWordsRemover(
    inputCol="words_raw", outputCol="words",
    stopWords=StopWordsRemover.loadDefaultStopWords("english") + STOPWORDS_FR
)
cv         = CountVectorizer(inputCol="words", outputCol="raw_features", vocabSize=3000, minDF=5.0)
idf        = IDF(inputCol="raw_features", outputCol="tfidf_features", minDocFreq=5)
normalizer = Normalizer(inputCol="tfidf_features", outputCol="features", p=2.0)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, cv, idf, normalizer])

print("Fitting TF-IDF pipeline...")
tfidf_model = tfidf_pipeline.fit(all_text)
tfidf_df    = tfidf_model.transform(all_text)
print(f"Vocabulaire final : {len(tfidf_model.stages[2].vocabulary):,} termes")
tfidf_df.select("text", "features").show(3, truncate=60)


In [ ]:
# ── A3. KMEANS CONTENU (k=8) ──────────────────────────────────────────────────
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

K_CONTENT = 15

kmeans_content = KMeans(
    featuresCol="features",
    predictionCol="content_cluster",
    k=K_CONTENT,
    seed=42,
    maxIter=50
)

print(f"Training K-Means content (k={K_CONTENT})...")
km_content_model = kmeans_content.fit(tfidf_df)
content_df = km_content_model.transform(tfidf_df)

# Silhouette score
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="content_cluster")
silhouette = evaluator.evaluate(content_df)
print(f"Silhouette Score (content): {silhouette:.4f}")

# Distribution par cluster
content_df.groupBy("content_cluster").count().orderBy("content_cluster").show()

In [ ]:
# ── A4. TOP TERMES PAR CLUSTER CONTENU ───────────────────────────────────────
import numpy as np
from pyspark.ml.stat import Summarizer

cv_model = tfidf_model.stages[2]
vocab    = cv_model.vocabulary

content_cluster_info = []

for cluster_id in range(K_CONTENT):
    cluster_subset = content_df.filter(F.col("content_cluster") == cluster_id)
    count = cluster_subset.count()

    top_platforms = (
        cluster_subset.groupBy("platform").count()
        .orderBy(F.desc("count")).limit(3)
        .select("platform").rdd.flatMap(lambda x: x).collect()
    )

    # Summarizer.mean aggrège les vecteurs sparse TF-IDF nativement
    mean_vec    = cluster_subset.select(Summarizer.mean(F.col("tfidf_features"))).collect()[0][0]
    top_indices = np.argsort(mean_vec.toArray())[::-1][:15]
    top_terms   = [vocab[i] for i in top_indices if i < len(vocab)]

    sample_texts = (
        cluster_subset.orderBy(F.desc("weight"))
        .select("text").limit(5)
        .rdd.flatMap(lambda x: x).collect()
    )

    content_cluster_info.append({
        "cluster_id":    cluster_id,
        "item_count":    count,
        "top_terms":     top_terms,
        "top_platforms": top_platforms,
        "sample_texts":  sample_texts
    })

    print(f"[Cluster {cluster_id}] {count:,} items | Plateformes: {top_platforms}")
    print(f"  Top termes : {', '.join(top_terms[:10])}")
    print(f"  Exemples   : {sample_texts[:3]}")


In [ ]:
# ── A5. LABELLING MANUEL DES CLUSTERS CONTENU ────────────────────────────────
# Labels basés sur le run k=15 (après filtres anti-pollution + Gmail + cap 2000/platform)
# NOTE : content clustering utilisé pour référence — home page s'appuiera
#        sur l'enrichissement CATEGORY_KEYWORDS + données Delta (Part B prime)

CONTENT_LABELS = {
    0:  {"label": "🌀 Divers",              "emoji": "🌀"},   # catch-all structurel
    1:  {"label": "🎵 Musique FR",          "emoji": "🎵"},
    2:  {"label": "💻 Extensions & Outils", "emoji": "💻"},
    3:  {"label": "🖥️ Tech & Gratuit",      "emoji": "🖥️"},
    4:  {"label": "🌃 Sorties Bruxelles",   "emoji": "🌃"},
    5:  {"label": "🛍️ Shopping",            "emoji": "🛍️"},
    6:  {"label": "🎌 Manga & Web",         "emoji": "🎌"},
    7:  {"label": "🍎 Apple & Apps",        "emoji": "🍎"},
    8:  {"label": "🌍 Belgique & Culture",  "emoji": "🌍"},
    9:  {"label": "🌀 Divers",              "emoji": "🌀"},
    10: {"label": "🎓 Études & Voyage",     "emoji": "🎓"},
    11: {"label": "📱 Apps & Productivité", "emoji": "📱"},
    12: {"label": "🌀 Divers",              "emoji": "🌀"},
    13: {"label": "✈️ Voyage",              "emoji": "✈️"},
    14: {"label": "🎮 Gaming",              "emoji": "🎮"},
}

print("Labels définis. Continuer vers A6.")


In [ ]:
# ── A6. ECRITURE content_clusters ─────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, LongType

rows = []
for info in content_cluster_info:
    cid = info["cluster_id"]
    rows.append((
        cid,
        CONTENT_LABELS.get(cid, {}).get("label", f"Cluster {cid}"),
        CONTENT_LABELS.get(cid, {}).get("emoji", "❓"),
        info["top_terms"],
        info["top_platforms"],
        info["sample_texts"],
        info["item_count"]
    ))

schema = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("top_keywords",  ArrayType(StringType()), True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("sample_items",  ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

content_clusters_df = spark.createDataFrame(rows, schema)

out_path = os.path.join(WAREHOUSE, "content_clusters")
content_clusters_df.write.mode("overwrite").parquet(out_path)

print(f"Ecrit dans : {out_path}")
content_clusters_df.show(truncate=60)

In [ ]:
spark.stop()
print("Spark session fermée. Notebook terminé.")